In [1]:
import torch
import timm
from torch.profiler import profile, record_function, ProfilerActivity

In [2]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("highest")
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
device

device(type='cuda')

In [5]:
model = timm.create_model("vit_tiny_patch16_224", pretrained=True).to(device)

In [6]:
BATCH_SIZE = 64
WARMUP_ITER = 10
PROFILER_ITER = 15

In [7]:
x = torch.randn(BATCH_SIZE, 3, 224, 224).to(device)

In [8]:
model.eval() # Critical for accurate inference profiling results
print("Warming up...")

with torch.inference_mode():
    for _ in range(WARMUP_ITER):
        with torch.autocast(device.type, dtype=torch.float16):
            _ = model(x)

if torch.cuda.is_available():
    torch.cuda.synchronize()

print("Warming up done. Starting profiling...")

Warming up...
Warming up done. Starting profiling...


In [9]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

In [10]:
print("Latency measurement")
starter = torch.cuda.Event(enable_timing=True)
ender = torch.cuda.Event(enable_timing=True)
torch.cuda.synchronize()
starter.record()
with torch.inference_mode():
    for _ in range(PROFILER_ITER ):
        with torch.autocast(device.type, dtype=torch.float16):
            _ = model(x)
ender.record()
torch.cuda.synchronize()

total_time_ms = starter.elapsed_time(ender)
avg_latency = total_time_ms / PROFILER_ITER

print(f"\n Average latency per batch: {avg_latency:.3f} msec")
print(f"Throughput: {(1000/avg_latency) * 8:.2f} samples/sec")

Latency measurement

 Average latency per batch: 21.023 msec
Throughput: 380.54 samples/sec


In [11]:
with torch.inference_mode():
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True, profile_memory=True, with_flops=True) as prof:
        with record_function("model_inference"):
            # for _ in range(PROFILER_ITER):
            with torch.autocast(device.type, dtype=torch.float16):
                _ = model(x)

if torch.cuda.is_available():
    torch.cuda.synchronize()
    print(f"Peak GPU memory usage: {torch.cuda.max_memory_allocated() / (1024 ** 2):.2f} MB") # Measures memory allocated by tensors by PyTorch, not total GPU memory usage
    torch.cuda.reset_peak_memory_stats()


Peak GPU memory usage: 151.88 MB


/home/somik/.local/lib/python3.10/site-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


In [ ]:
prof.export_chrome_trace("trace.json")


In [ ]:
evt = prof.key_averages()[0]
print([attr for attr in dir(evt) if not attr.startswith("_")])

In [ ]:
print("Printing all events with their attributes:")
print(prof.events()) 

In [ ]:
print(prof.key_averages())

In [ ]:
# for evt in prof.events():
#     if evt.input_shapes:
#         print(evt.name, evt.input_shapes)


In [ ]:
# print(prof.key_averages())

In [ ]:
print(prof.key_averages().table(sort_by="self_device_time_total"))

In [ ]:
print(prof.key_averages().table(sort_by="device_time_total"))

In [ ]:
print(prof.key_averages().table(sort_by="self_cuda_memory_usage"))

In [12]:
print(prof.key_averages().table(sort_by="self_cuda_time_total"))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  Total MFLOPs  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                        model_inference         0.00%       0.000us         0.00%       0.000us       0.000us      18.952ms       118.23%      18.952ms      18.952ms           0 

In [12]:
print(prof.key_averages().table(sort_by="self_cuda_time_total"))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  Total MFLOPs  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                        model_inference         0.00%       0.000us         0.00%       0.000us       0.000us      44.526ms       101.38%      44.526ms      44.526ms           0 

In [12]:
print(prof.key_averages().table(sort_by="self_cuda_time_total"))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  Total MFLOPs  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                        model_inference         0.00%       0.000us         0.00%       0.000us       0.000us      44.565ms       100.53%      44.565ms      44.565ms           0 

In [12]:
print(prof.key_averages().table(sort_by="self_cuda_time_total"))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  Total MFLOPs  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                        model_inference         0.00%       0.000us         0.00%       0.000us       0.000us      17.282ms       107.57%      17.282ms      17.282ms           0 

In [ ]:
print(
    prof.key_averages()
        .table(sort_by="self_device_memory_usage", row_limit=-1)
)


In [ ]:
for evt in prof.key_averages():
    if evt.device_memory_usage != 0:
        print(f"{evt.key}")
        print(f"  Total GPU Memory: {evt.device_memory_usage / 1024**2:.2f} MB")
        print(f"  Self GPU Memory: {evt.self_device_memory_usage / 1024**2:.2f} MB")
        print("-" * 40)


In [ ]:
for evt in prof.key_averages():
    if evt.device_time_total != 0:
        print(f"{evt.key}")
        print(f"  Total GPU Time: {evt.device_time_total/1000:.3f} ms")
        print(f"  Self GPU Time: {evt.self_device_time_total/1000:.3f} ms")
        print("-" * 40)